# Radial Basis Nearest Neighbor (RBNN)

- Giulia Monteiro Garrido (RA: 24010281)
- Mateus Antezana da Silva (RA: )
- Vitor Furuta da Silva (RA: 24008775)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Sequence

In [3]:
# def RBNN()
class Distancias:
    def __init__(self): #tem que ter __init__ para funfar
        pass

    def _validar_vetores(self, A:Sequence[int | float], B:Sequence[int | float]):
        A = np.asarray(A, dtype=float)
        B = np.asarray(B, dtype=float)

        if A.shape != B.shape:
            raise ValueError("Os vetores devem ter a mesma dimensão.")

        if A.size == 0:
            raise ValueError("Os vetores não podem estar vazios.")

        return A, B

    def distancia(self, funcao, A:Sequence[int | float], B:Sequence[int | float]) -> int | float | None:
        """
        Args:
            funcao (callable): recebe outras funções como parametro, aceitando apenas MINKOWSKI,
                COSSENO, EUCLIDIANA e  MANHATTAN.
            A & B (Sequence[int | float]): array numérico podendo conter inteiros e decimais de mesma dimensão.
        Retorno:
            int | float | None
        >>>var.distancia(funcao = var.manhattan, A = [1,3,2,3], B = [0,1,1,1])
        6        
        """

        try:
            A, B = self._validar_vetores(A, B)
            return funcao(A, B)
        
        except IndexError:
            print(IndexError)
            
        except ValueError:
            print(ValueError)

    def minkowski(self, A:Sequence[int | float], B:Sequence[int | float], p:int) -> float | int | None:
        """
        Args:
            A & B (Sequence[int | float]): array numérico podendo conter inteiros e decimais de mesma dimensão.
            p (int): valor do parâmetro p da distância de Minkowski.
        Retorno:
            int | float | None
        """

        try: 
            
            if p <= 0:
                raise ValueError("p deve ser maior que zero.")
            
            A, B = self._validar_vetores(A, B)

            return sum(abs(A[i]-B[i])**p for i in range(len(A))) ** (1/p)

        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)
        

    def cosseno(self, A:Sequence[int | float], B:Sequence[int | float]) -> float | int | None:
        """
        Args:
            A & B (Sequence[int | float]): array numérico podendo conter inteiros e decimais de mesma dimensão.
        Retorno:
                int | float | None
        """
        try:
            A, B = self._validar_vetores(A, B)

            produto = np.dot(A, B)
            norma_A = np.linalg.norm(A)
            norma_B = np.linalg.norm(B)

            if norma_A == 0 or norma_B == 0:
                return 0

            return float(1 - (produto/ (norma_A * norma_B)))
        
        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)
        
    def manhattan(self, A:Sequence[int | float], B:Sequence[int | float]) -> float | int | None:
        """
                Args:
                    A & B (Sequence[int | float]): array numérico podendo conter inteiros e decimais de mesma dimensão.
                Retorno:
                        int | float | None
        """
        try:
            A,B = self._validar_vetores(A,B)
            return sum((abs(A[i]-B[i])) for i in range(len(A)))

    
        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)
    
    def euclidiana(self, A:Sequence[int | float], B:Sequence[int | float]) -> float | int | None:
        """
                Args:
                    A & B (Sequence[int | float]): array numérico podendo conter inteiros e decimais de mesma dimensão.
                Retorno:
                        int | float | None
        """
        try:
            A,B = self._validar_vetores(A,B)
            return sum(((A[i])-(B[i])**2) for i in range(len(A)))**(1/2)

        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)

In [ ]:
def pesos(x, X_treino, sigma, distancia):
    """
    Calcula o peso gaussiano de x em relação a cada ponto de treino,
    com base na distância escolhida e no sigma escolhido.
    """
    distancias = np.array([
        distancia(x, xi) for xi in X_treino
    ])
    formula = 1 / (sigma * np.sqrt(2 * np.pi))
    pesos = formula * np.exp(-(distancias ** 2) / (2 * sigma ** 2))
    return pesos


def regressao(x, X_treino, y_treino, sigma, distancia):
    """
    Previsão por média ponderada usando pesos gaussianos.
    Args:
        x (array[int | float]): Ponto a ser previsto.
        X_treino (array[int | float]): Pontos de treino.
        y_treino (array[int | float]): Valores de treino.
        sigma (float): Parâmetro de suavização.
        distancia (function): Função de distância.
    Returns:
        float: Valor previsto.
    """
    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:
        # se peso 0 cai para o vizinho mais próximo
        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    return np.sum(w * y_treino) / np.sum(w)

def classificacao(x, X_treino, y_treino, sigma, distancia):
    """
    Previsão por votação ponderada, soma os pesos gaussianos por classe
    e prevê a classe com maior soma.

    Args:
        x (array[int | float]): Ponto a ser previsto.
        X_treino (array[int | float]): Pontos de treino.
        y_treino (array[int | float]): Classes de treino.
        sigma (float): Parâmetro de suavização.
        distancia (function): Função de distância.
    Returns:
        int | float: Classe prevista.
    """
    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:
        # se peso 0 cai para o vizinho mais próximo
        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    y_treino = np.asarray(y_treino)
    classes = np.unique(y_treino)
    votos = {c: w[y_treino == c].sum() for c in classes}
    return max(votos, key=votos.get)
